# Metrics and evaluation

**Goal:** learn to judge a classifier the way this problem demands: rare real objects, thousands of bogus ones, and a
human who must vet every alert.

We evaluate a simple baseline: logistic regression on 6 measured numbers per detection. It's also the score the CNN must beat later.

Read `docs/metrics_evaluation.md` alongside this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve

from asthunt.config.load_config import load_config
from asthunt.dataset.make_unit_tag import make_unit_tag
from asthunt.acquire.load_unit_metadata import load_unit_metadata
from asthunt.models.train_baseline_classifier import train_baseline_classifier
from asthunt.dataset.balance_classes import balance_classes
from asthunt.evaluate.compute_classification_metrics import compute_classification_metrics
from asthunt.evaluate.choose_threshold import choose_threshold
from asthunt.evaluate.calibrate_probabilities import calibrate_probabilities
from asthunt.evaluate.adjust_for_base_rate import adjust_for_base_rate
from asthunt.evaluate.measure_completeness import measure_completeness

project = load_config("project"); paths = project["paths"]
acquire, dataset = load_config("acquire"), load_config("dataset")


def unit_of(night):
    """The original quadrant (configs/acquire.yaml: field, ccd, quadrant) on one night."""
    return {"field": acquire["field"], "ccd": acquire["ccd"], "quadrant": acquire["quadrant"], "night": night}


rng = np.random.default_rng(project["seed"])


def load_nights(nights):
    tables = [pd.read_parquet(paths["processed"] / f"detections__{make_unit_tag(unit_of(n))}.parquet") for n in nights]
    return pd.concat(tables, ignore_index=True)


train_all = load_nights(acquire["nights"]["train"])
validation = load_nights(acquire["nights"]["validation"])

FEATURES = ["snr", "peak", "npix", "a", "b", "flux"]
train = train_all.iloc[balance_classes(train_all["label"], dataset["bogus_per_real"], rng)]
model = train_baseline_classifier(train[FEATURES].to_numpy(), train["label"].to_numpy())

validation_scores = model.predict_proba(validation[FEATURES].to_numpy())[:, 1]
validation_labels = validation["label"].to_numpy()
print(f"validation: {len(validation)} detections, {validation_labels.sum()} real ({validation_labels.mean():.2%})")

## 1. Accuracy is misleading here
A "model" that always says *bogus* is compared with our baseline:

In [ ]:
always_bogus = compute_classification_metrics(validation_labels, np.zeros(len(validation_labels)), threshold=0.5)
baseline = compute_classification_metrics(validation_labels, validation_scores, threshold=0.5)
print(f"always 'bogus': accuracy {always_bogus['accuracy']:.3f}, recall {always_bogus['recall']:.3f}")
print(f"baseline      : accuracy {baseline['accuracy']:.3f}, recall {baseline['recall']:.3f}, precision {baseline['precision']:.3f}")

The do-nothing model scores **96.8%** accuracy and finds **zero** asteroids; the baseline scores 98.4%. When one class
is rare, accuracy mostly measures the common class, so the two look almost equally good. We need **recall** (completeness)
and **precision** (purity).

## 2. The confusion matrix at threshold 0.5

In [ ]:
m = baseline
print(f"                 called real   called bogus")
print(f"truly real    {m['true_positives']:12d} {m['false_negatives']:13d}")
print(f"truly bogus   {m['false_positives']:12d} {m['true_negatives']:13d}")
n_frames = validation.groupby(["night", "frame_index"]).ngroups
print(f"\nfalse positives per frame: {m['false_positives'] / n_frames:.0f}  <- a human would have to look at these")

## 3. ROC vs precision-recall curves
Both sweep the threshold from 1 to 0.
- **ROC:** recall vs false-positive *rate* (FP / all bogus). With 35,000 bogus, a 1% rate is still 350 false alarms, but looks tiny on the plot.
- **Precision-recall:** recall vs precision (TP / all alerts). It shows directly how polluted the alerts are. **Use this one for rare classes.**

In [ ]:
fpr, tpr, _ = roc_curve(validation_labels, validation_scores)
precision, recall, _ = precision_recall_curve(validation_labels, validation_scores)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(fpr, tpr); axes[0].plot([0, 1], [0, 1], "--", c="gray")
axes[0].set_xlabel("false positive rate"); axes[0].set_ylabel("recall"); axes[0].set_title(f"ROC   (AUC = {m['roc_auc']:.3f})")
axes[1].plot(recall, precision); axes[1].axhline(validation_labels.mean(), ls="--", c="gray", label="random guessing")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision"); axes[1].set_title(f"precision-recall   (AP = {m['average_precision']:.3f})")
axes[1].legend(); plt.show()

The ROC AUC looks excellent. The precision-recall curve tells the honest story: keeping high recall costs a lot of purity.

## 4. Choosing a threshold (on validation data, never test)
Our rule: keep **90% of real detections**, and accept the false alarms that come with it.

In [ ]:
threshold = choose_threshold(validation_labels, validation_scores, target_recall=0.90)
at_90 = compute_classification_metrics(validation_labels, validation_scores, threshold)
print(f"threshold {threshold:.3f}: recall {at_90['recall']:.3f}, precision {at_90['precision']:.3f}, "
      f"false positives per frame {at_90['false_positives'] / n_frames:.0f}")

## 5. Are the scores probabilities? Calibration
If a model says 0.8 for many detections, about 80% of them should be real. Our model was trained on 1 real : 5 bogus,
but validation has about 1 : 35. Its scores are too optimistic.

In [ ]:
def reliability(scores, labels, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    which = np.clip(np.digitize(scores, edges) - 1, 0, bins - 1)
    centres, observed = [], []
    for b in range(bins):
        inside = which == b
        if inside.sum() >= 20:
            centres.append(scores[inside].mean()); observed.append(labels[inside].mean())
    return centres, observed

train_real_fraction = train["label"].mean()
true_real_fraction = validation_labels.mean()
adjusted = adjust_for_base_rate(validation_scores, train_real_fraction, true_real_fraction)

# Fit the calibrator on HALF of validation, check it on the other half (no peeking)
half = rng.permutation(len(validation_scores)) < len(validation_scores) // 2
calibrator = calibrate_probabilities(validation_scores[half], validation_labels[half])
calibrated = calibrator.predict(validation_scores[~half])

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], "--", c="gray", label="perfect")
plt.plot(*reliability(validation_scores[~half], validation_labels[~half]), "o-", label="raw model")
plt.plot(*reliability(adjusted[~half], validation_labels[~half]), "s-", label="base-rate adjusted")
plt.plot(*reliability(calibrated, validation_labels[~half]), "^-", label="isotonic calibrated")
plt.xlabel("predicted probability"); plt.ylabel("observed fraction real"); plt.legend(); plt.show()

Raw scores sit far below the diagonal: overconfident. Both fixes pull them towards honesty:
- **Base-rate adjustment** uses only arithmetic (Bayes' rule) and the two class fractions.
- **Isotonic calibration** learns the mapping from data.

Calibration does **not** change the ranking, so recall and precision at a matching threshold stay the same. It changes what
the numbers *mean*, which matters when you tell someone "this candidate has a 70% chance of being real".

## 6. Completeness vs magnitude: the astronomer's view
Recall per brightness bin, for injected fakes only, at the 90%-recall threshold.

In [ ]:
injections = pd.concat([pd.read_parquet(paths["processed"] / f"injections__{make_unit_tag(unit_of(n))}.parquet")
                        for n in acquire["nights"]["validation"]], ignore_index=True)

validation["score"] = validation_scores
fakes_found = validation[validation["injection_id"] >= 0]
detected_keys = set(zip(fakes_found["night"], fakes_found["injection_id"], fakes_found["frame_index"]))
kept_keys = set(zip(*[fakes_found.loc[fakes_found["score"] >= threshold, c] for c in ["night", "injection_id", "frame_index"]]))

injections["detected"] = [k in detected_keys for k in zip(injections["night"], injections["injection_id"], injections["frame_index"])]
injections["kept"] = [k in kept_keys for k in zip(injections["night"], injections["injection_id"], injections["frame_index"])]

bins = np.arange(17, 22.25, 0.25)
detected_curve = measure_completeness(injections, "detected", "mag", bins)
kept_curve = measure_completeness(injections, "kept", "mag", bins)
plt.errorbar(detected_curve["bin_centre"], detected_curve["completeness"], detected_curve["uncertainty"], fmt="o-", label="detected")
plt.errorbar(kept_curve["bin_centre"], kept_curve["completeness"], kept_curve["uncertainty"], fmt="s-", label="detected AND kept by classifier")
plt.xlabel("injected magnitude"); plt.ylabel("completeness"); plt.ylim(0, 1); plt.legend(); plt.show()

The gap between the curves is what the classifier throws away. It's widest at the faint end: exactly where we want to
find new asteroids. A better classifier (the CNN) should close that gap.

## Exercises
1. Choose the threshold for 95% recall instead. How many false positives per frame does that cost?
2. Train the baseline on all training rows (no balancing). How do the raw scores and the reliability plot change?
3. Why did we fit the calibrator on half of validation and check it on the other half?